In [4]:
# diffpool_pretrain.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, global_mean_pool
from torch_geometric.nn.dense import dense_diff_pool
from torch_geometric.utils import to_dense_adj, to_dense_batch
from torch_geometric.data import DataLoader


# -------------------------
# Utilities: augmentations
# -------------------------
def edge_dropout(adj, drop_prob):
    # adj: (B, N, N) dense adjacency (0/1 or weights)
    if drop_prob <= 0:
        return adj
    mask = (torch.rand_like(adj) > drop_prob).float()
    # keep symmetry
    mask = ((mask + mask.transpose(-1, -2)) > 0).float()
    return adj * mask


def feature_mask(x, mask_prob):
    # x: (B, N, F)
    if mask_prob <= 0:
        return x
    mask = (torch.rand(x.shape[:2], device=x.device) > mask_prob).unsqueeze(-1)  # (B,N,1)
    return x * mask


# -------------------------
# NT-Xent (in-batch) loss
# -------------------------
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
        # z1, z2: (M, D) where M is total number of cluster embeddings in the batch per view
        # We assume z1[i] corresponds to positive z2[i]
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)
        representations = torch.cat([z1, z2], dim=0)  # (2M, D)
        similarity_matrix = torch.matmul(representations, representations.t())  # (2M,2M)

        # create labels
        M = z1.size(0)
        diag = torch.arange(M, device=z1.device)
        positives = torch.cat([diag + M, diag], dim=0)  # index of positive for each sample in stacked
        logits = similarity_matrix / self.temperature

        # mask to remove self-similarity
        mask = (~torch.eye(2 * M, dtype=bool, device=z1.device)).float()

        # for numerical stability: subtract max
        logits_max, _ = torch.max(logits * mask + (1 - mask) * -9e15, dim=1, keepdim=True)
        logits = logits - logits_max.detach()

        exp_logits = torch.exp(logits) * mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

        # positive logits (for stacked index i, the positive index is positives[i])
        positives_idx = positives
        loss = -log_prob[torch.arange(2 * M, device=z1.device), positives_idx]
        return loss.mean()


# -------------------------
# DiffPool model
# -------------------------
class EmbedNet(nn.Module):
    def __init__(self, in_channels, hidden):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, hidden)

    def forward(self, x, edge_index):
        # For dense usage we'll accept x and edge_index and then use sparse convs on the original graph.
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return x


class PoolNet(nn.Module):
    def __init__(self, in_channels, hidden, assign_size):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, assign_size)  # outputs logits for assignment

    def forward(self, x, edge_index):
        a = F.relu(self.conv1(x, edge_index))
        a = self.conv2(a, edge_index)
        return F.softmax(a, dim=-1)  # row-wise softmax on nodes -> clusters


class DiffPoolPretrainModel(nn.Module):
    def __init__(self, in_channels, hidden, assign_size, num_classes=None):
        super().__init__()
        self.embed_gnn1 = EmbedNet(in_channels, hidden)
        self.pool_gnn1 = PoolNet(in_channels, hidden, assign_size)
        # after pooling we can add a small embedding GNN for coarser levels (optional)
        self.post_embed = nn.Linear(hidden, hidden)  # projection for cluster embeddings

    def forward(self, x, edge_index, batch, adj_dense=None, mask=None):
        # run embedding and pool networks (sparse convs)
        z = self.embed_gnn1(x, edge_index)  # (total_nodes, H)
        s_logits = self.pool_gnn1(x, edge_index)  # (total_nodes, C)
        # gather dense batches
        z_batch, mask = to_dense_batch(z, batch)  # (B, N, H)
        s_batch = to_dense_batch(s_logits, batch)[0]  # (B, N, C)
        # optionally convert to adj_dense if needed externally
        if adj_dense is None:
            # compute adjacency from edge_index -> dense
            adj_dense = to_dense_adj(edge_index, batch)  # (B, N, N)
        x_pooled, adj_pooled, lpool, e = dense_diff_pool(z_batch, adj_dense, s_batch, mask)
        # x_pooled: (B, C, H) cluster embeddings
        # For contrastive, flatten cluster embeddings across batch: (B*C, H)
        B, C, H = x_pooled.size()
        clusters = x_pooled.view(B * C, H)
        clusters = self.post_embed(clusters)  # projection head
        return clusters, s_batch, z_batch, adj_dense, mask


# -------------------------
# Training routine
# -------------------------
def pretrain_loop(model, dataloader, optimizer, device,
                  epochs=100, augment_p=0.2, mask_p=0.1,
                  alpha_lp=10.0, beta_ent=0.1, temperature=0.1):
    model.train()
    nt_xent = NTXentLoss(temperature).to(device)
    for epoch in range(epochs):
        total_loss = 0.0
        lp_total_loss = 0.0
        ent_total_loss = 0.0
        for data in dataloader:
            data = data.to(device)
            # build dense adjacency and batch info
            edge_index = data.edge_index
            batch = data.batch
            x = data.x

            # create two augmented views
            # 1) original dense adjacency
            adj = to_dense_adj(edge_index, batch)  # (B,N,N)
            x_dense, mask = to_dense_batch(x, batch)  # (B,N,F)

            # view A
            adj_a = edge_dropout(adj, augment_p)
            x_a = feature_mask(x_dense, mask_p)

            # view B
            adj_b = edge_dropout(adj, augment_p)
            x_b = feature_mask(x_dense, mask_p)

            # We need sparse edge_index for forward convs.
            # Simplest: use original edge_index for convs; augmentations used only for dense_diff_pool inputs.
            # Forward view A
            clusters_a, s_a, z_a, adj_used_a, mask_a = model(x.view(-1, x.size(-1)), edge_index, batch, adj_dense=adj_a)
            # Forward view B
            clusters_b, s_b, z_b, adj_used_b, mask_b = model(x.view(-1, x.size(-1)), edge_index, batch, adj_dense=adj_b)

            # Contrastive loss between clusters (matches by index: assume same C per graph)
            # Because some graphs may have fewer nodes, some clusters might be unused (low norm) -> it's OK.
            # Only keep clusters corresponding to real graphs: mask indicates nodes used; we assume fixed C per graph.
            # For simplicity, we treat all B*C clusters as positive-aligned by position.
            loss_cl = nt_xent(clusters_a, clusters_b)

            # Link-prediction loss: ||A - S S^T||_F per graph, averaged
            # s_a: (B,N,C) soft assignments. compute S S^T -> (B, N, N) reconstructed adjacency
            SS_t = torch.matmul(s_a, s_a.transpose(1, 2))
            # adj a is (B,N,N)
            lp_loss = F.mse_loss(SS_t * mask_a.unsqueeze(1).float(), adj_used_a.float())

            # Entropy regularizer: encourage near one-hot rows of S
            # compute row entropy across clusters
            ent = - (s_a * (s_a + 1e-12).log()).sum(dim=-1)  # (B,N)
            ent_loss = ent.mean()

            loss = loss_cl + alpha_lp * lp_loss + beta_ent * ent_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            lp_total_loss += alpha_lp * lp_loss.item() * data.num_graphs
            ent_total_loss += beta_ent * ent_loss.item() * data.num_graphs

            total_loss += loss.item() * data.num_graphs

            with torch.no_grad():
                B, N, C = s_a.shape
                cluster_usage = s_a.sum(dim=1).mean(dim=0)        # average cluster mass
                unused_frac = (cluster_usage < 1e-3).float().mean().item()

                B, C, H = clusters_a.view(B, C, -1).shape
                clusters_flat = clusters_a.view(B*C, -1)
                norms = clusters_flat.norm(dim=1)
                mean_norm = norms.mean().item()
                clusters_normed = F.normalize(clusters_flat, dim=1)
                sim = torch.mm(clusters_normed, clusters_normed.t())
                off_diag = sim[~torch.eye(sim.size(0), dtype=bool, device=sim.device)]
                mean_offdiag_sim = off_diag.mean().item()


                print(f"[Diag] unused_frac={unused_frac:.2f}, mean_norm={mean_norm:.2f}, mean_offdiag_sim={mean_offdiag_sim:.2f}")
        print(
            f"Epoch {epoch:03d}, Loss {total_loss / len(dataloader.dataset):.4f}, LP={lp_total_loss/ len(dataloader.dataset):.4f}, Entropy={ent_total_loss/ len(dataloader.dataset):.4f}")


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    from torch_geometric.datasets import TUDataset

    dataset = TUDataset(root='data/TUD', name='PROTEINS')  # replace with a large unlabeled dataset ideally
    loader = DataLoader(dataset, batch_size=16, shuffle=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    in_ch = dataset.num_node_features
    hidden = 128
    assign_size = 16  # clusters per graph
    model = DiffPoolPretrainModel(in_ch, hidden, assign_size).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

    pretrain_loop(model, loader, opt, device, epochs=50)


[Diag] unused_frac=0.00, mean_norm=1.96, mean_offdiag_sim=0.90
[Diag] unused_frac=0.00, mean_norm=2.21, mean_offdiag_sim=0.71
[Diag] unused_frac=0.00, mean_norm=1.45, mean_offdiag_sim=0.56
[Diag] unused_frac=0.00, mean_norm=1.57, mean_offdiag_sim=0.51
[Diag] unused_frac=0.00, mean_norm=1.42, mean_offdiag_sim=0.45
[Diag] unused_frac=0.00, mean_norm=1.03, mean_offdiag_sim=0.32
[Diag] unused_frac=0.00, mean_norm=1.49, mean_offdiag_sim=0.36
[Diag] unused_frac=0.00, mean_norm=1.45, mean_offdiag_sim=0.29
[Diag] unused_frac=0.00, mean_norm=1.30, mean_offdiag_sim=0.31
[Diag] unused_frac=0.00, mean_norm=1.00, mean_offdiag_sim=0.23
[Diag] unused_frac=0.00, mean_norm=1.14, mean_offdiag_sim=0.22
[Diag] unused_frac=0.00, mean_norm=1.47, mean_offdiag_sim=0.23
[Diag] unused_frac=0.00, mean_norm=1.15, mean_offdiag_sim=0.17
[Diag] unused_frac=0.00, mean_norm=1.07, mean_offdiag_sim=0.20
[Diag] unused_frac=0.00, mean_norm=1.46, mean_offdiag_sim=0.16
[Diag] unused_frac=0.00, mean_norm=1.46, mean_offdiag_s

KeyboardInterrupt: 